# Projeto 1 — Introdução à Inteligência Artificial - Sistema de Recomendação de Jogos

## Integrantes:

- Maylla Krislainy — 190043873
- Mateus Valério — 190035161


---

# Introdução e Metodologia

Este projeto implementa um **Sistema de Recomendação de Jogos** baseado na abordagem de **Filtragem Baseada em Conteúdo** (Content-Based Filtering).

### Metodologia Aplicada:

- **Vetorização:** Utilizamos **One-Hot Encoding** (via `CountVectorizer`) para representar os atributos categóricos (Gênero, Perspectiva, Categoria) em vetores de 12 dimensões.
- **Perfil do Utilizador:** Construído através da média ponderada dos vetores dos jogos avaliados, utilizando as notas (1-5) como pesos.
- **Algoritmo:** Cálculo de **Similaridade do Cosseno** entre o perfil do utilizador e os itens do catálogo, aplicando a lógica de **K-Nearest Neighbors (KNN)** para retornar o Top-K.


---

### Repositório e Código-Fonte

Este notebook foca na construção dos dados e no motor matemático do modelo.
A aplicação completa, que inclui uma interface interativa em Angular e o backend em Python, pode ser acessada em nosso repositório:
👉 **[Acesse o repositório do projeto no GitHub aqui](https://github.com/im-maV/iia-recommendation-app/tree/main)**


---

# Dependências e Imports

### Configuração do Ambiente

Para garantir a correta execução do projeto e evitar conflitos de dependências, recomenda-se a utilização de um ambiente virtual Python (`venv`).
Após ativar o ambiente virtual, instale os pacotes necessários:

`pip install -r requirements.txt`

As seguintes bibliotecas são necessárias para a execução dos códigos:

- fastapi[standard]
- uvicorn
- pydantic
- scikit-learn
- numpy
- pandas
- ijson
- notebook
- jupyterlab


---

# Construção do Dataset de Jogos

O conjunto de dados utilizado no projeto foi construído a partir de um dataset público de jogos da plataforma Steam. Como o dataset original possui milhares de jogos e informações não estruturadas, foi necessário desenvolver um processo de filtragem e seleção para adequá-lo ao escopo do trabalho.


---

### 1. Definição do Vocabulário de Características

Inicialmente, foram definidas as possíveis características que cada jogo poderia possuir:

- **Gênero**: Survival Horror, RPG, Shooter, Simulation, Survival, Souls-like
- **Perspectiva**: First-Person, Third Person, 2D
- **Categoria**: Singleplayer, Co-op, Multiplayer

Essas categorias são utilizadas como base para a representação vetorial dos jogos no modelo de recomendação.


---

### 2. Filtragem e Normalização dos Dados

O dataset original é percorrido utilizando a biblioteca `ijson`, permitindo a leitura incremental do arquivo (streaming), o que é mais eficiente para grandes volumes de dados.

Durante essa etapa:

- Cada jogo tem seu número de donos (`estimated_owners`) convertido para um valor numérico
- As tags do jogo são analisadas para identificar:
  - um gênero válido
  - uma perspectiva válida
  - uma categoria válida
- Jogos que não possuem ao menos uma característica válida em cada dimensão são descartados

Ao final, obtém-se um subconjunto de jogos consistentes, contendo exatamente uma característica por dimensão.


In [1]:
import os, json, re, random
import ijson
from collections import defaultdict
from pathlib import Path


BASE_DIR = Path().resolve()
DEST_FILE_PATH = BASE_DIR / "50_games.json"
SRC_FILE_PATH = BASE_DIR / "games.json"
FILTERED_FILE_PATH = BASE_DIR / "games_filtered.json"

# ── Vocabulário de caracteristicas
VALID_GENRES = {
    "Survival Horror",
    "RPG",
    "Shooter",
    "Simulation",
    "Survival",
    "Souls-like",
}
VALID_PERSPECTIVES = {"First-Person", "Third Person", "2D"}
VALID_CATEGORIES = {"Singleplayer", "Co-op", "Multiplayer"}


# ex.: "2000000 - 5000000" -> 2_000_000
def parse_owners(owners_str: str) -> int:
    try:
        return int(owners_str.split("-")[0].strip().replace(",", ""))
    except:
        return 0


# ── Normalização: mantém 1 valor por característica ───────────────────────
def normalize_game(game: dict, owners: int) -> dict | None:
    tags = list(game.get("tags", []))
    genre = next((g for g in tags if g in VALID_GENRES), None)
    persp = next((p for p in tags if p in VALID_PERSPECTIVES), None)
    cat = next((c for c in tags if c in VALID_CATEGORIES), None)

    if not (genre and persp and cat):
        return None

    return {
        "id": game["id"],
        "name": game["name"],
        "genre": genre,
        "perspective": persp,
        "category": cat,
        "owners": owners,  # usado para selecão, descartado depois
        "recommendations": game.get("recommendations", 0),
    }

---

### 3. Seleção dos 50 Jogos

Após a filtragem, ainda restam milhares de jogos. Portanto, foi necessário reduzir o conjunto para 50 itens, conforme especificado no projeto.

A estratégia adotada foi:

#### 🔹 Jogos Populares (~40)

- Selecionados com base no número de donos (`owners`)
- Ordenados de forma decrescente
- Amostrados de forma balanceada entre os gêneros

#### 🔹 Jogos Variados (~10)

- Selecionados aleatoriamente a partir dos jogos restantes
- Também balanceados por gênero

Essa abordagem garante:

- presença de jogos conhecidos (melhor experiência para o usuário)
- diversidade no catálogo (evita viés excessivo)


In [2]:
def filter_games(src=SRC_FILE_PATH, dst="games_filtered.json"):
    with open(src, "r", encoding="utf-8") as f_in, \
         open(dst, "w", encoding="utf-8") as f_out:

        objects = ijson.kvitems(f_in, "")
        f_out.write("[\n")
        first = True

        for app, raw in objects:
            owners  = parse_owners(raw.get("estimated_owners", "0"))
            game    = {**raw, "id": app}
            normed  = normalize_game(game, owners)
            if not normed:
                continue

            if not first:
                f_out.write(",\n")
            first = False
            json.dump(normed, f_out, ensure_ascii=False)

        f_out.write("\n]")
    print("Filtrado salvo em", dst)

---

### 4. Balanceamento por Gênero

Tanto na seleção de jogos populares quanto na seleção aleatória, foi utilizada uma estratégia de **amostragem estratificada por gênero**.

Isso evita que o dataset final fique desbalanceado (por exemplo, com excesso de jogos de um único gênero), garantindo uma distribuição mais uniforme das características.


---

### 5. Limpeza Final dos Dados

Após a seleção:

- atributos auxiliares como `owners` e `recommendations` são removidos
- apenas os campos relevantes são mantidos:
  - id
  - nome
  - gênero
  - perspectiva
  - categoria

O resultado final é salvo no arquivo `50_games.json`(presente no root deste projeto), que será utilizado pelo sistema de recomendação.


In [3]:
def build_balanced_dataset(
    src="games_filtered.json", dst="50_games.json", total=50, popular_threshold=500_000
):

    with open(src, encoding="utf-8") as f:
        games = json.load(f)

    print(f"Jogos válidos disponíveis: {len(games)}")

    half = int(total // 1.2)  # 40 aprox

    # 40 populares: ordena por owners desc, amostra por gênero
    popular = [g for g in games if g["owners"] >= popular_threshold]
    popular.sort(key=lambda x: x["owners"], reverse=True)

    # amostragem estratificada dentro dos populares
    pop_by_genre = defaultdict(list)
    for g in popular:
        pop_by_genre[g["genre"]].append(g)

    selected_popular = []
    genres = list(pop_by_genre.keys())
    per_genre = half // len(genres)

    for genre in genres:
        selected_popular.extend(pop_by_genre[genre][:per_genre])

    # completa até 25 se algum gênero tiver poucos jogos populares
    if len(selected_popular) < half:
        used_ids = {g["id"] for g in selected_popular}
        extras = [g for g in popular if g["id"] not in used_ids]
        selected_popular.extend(extras[: half - len(selected_popular)])

    # 25 variados: todos os não-populares, amostragem estratificada
    used_ids = {g["id"] for g in selected_popular}
    remaining = [g for g in games if g["id"] not in used_ids]

    rem_by_genre = defaultdict(list)
    for g in remaining:
        rem_by_genre[g["genre"]].append(g)

    selected_random = []
    for genre in rem_by_genre:
        pool = rem_by_genre[genre]
        random.shuffle(pool)
        selected_random.extend(pool[:per_genre])

    if len(selected_random) < half:
        used_ids2 = {g["id"] for g in selected_random}
        extras2 = [g for g in remaining if g["id"] not in used_ids2]
        random.shuffle(extras2)
        selected_random.extend(extras2[: half - len(selected_random)])

    # Combina e remove coluna auxiliar 'owners' do JSON final
    final = selected_popular[:half] + selected_random[:half]
    final = final[:total]

    for g in final:
        g.pop("owners", None)
        g.pop("recommendations", None)

    with open(dst, "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)

    print(f"Salvos {len(final)} jogos em {dst}")
    print(f"  Populares : {len(selected_popular[:half])}")
    print(f"  Variados  : {len(selected_random[:half])}")
    return final


# Execução
if SRC_FILE_PATH.exists():
    filter_games()

final_games = build_balanced_dataset(popular_threshold=500_000)

print("\nAmostra dos 5 primeiros:")
for g in final_games[:5]:
    print(g)

Jogos válidos disponíveis: 19126
Salvos 50 jogos em 50_games.json
  Populares : 41
  Variados  : 41

Amostra dos 5 primeiros:
{'id': '578080', 'name': 'PUBG: BATTLEGROUNDS', 'genre': 'Survival', 'perspective': 'First-Person', 'category': 'Multiplayer'}
{'id': '304930', 'name': 'Unturned', 'genre': 'Survival', 'perspective': 'First-Person', 'category': 'Multiplayer'}
{'id': '252490', 'name': 'Rust', 'genre': 'Survival', 'perspective': 'First-Person', 'category': 'Multiplayer'}
{'id': '105600', 'name': 'Terraria', 'genre': 'Survival', 'perspective': '2D', 'category': 'Multiplayer'}
{'id': '346110', 'name': 'ARK: Survival Evolved', 'genre': 'Survival', 'perspective': 'First-Person', 'category': 'Multiplayer'}


---

# Gerador da Matriz de Utilidade

Centraliza os perfis de usuário, parâmetros das distribuições de notas e funções de geração da matriz de utilidade.

**Sobre os perfis:**

- `genre`/`perspective`/`category`: `None` significa "aceita qualquer valor".
- Perfis com múltiplos gêneros simulam jogadores ecléticos.
- `n`: número de usuários deste perfil (a soma deve ser `N_USERS`).

**Sobre as distribuições:**

- **high**: o jogo bate bem com o perfil → nota 4-5
- **medium**: o jogo bate parcialmente → nota 3-4
- **low**: o jogo não combina → nota 1-2


---

### Etapa 1: Configurações e Perfis

Nesta primeira etapa, definimos as "regras do jogo": constantes gerais, os perfis (arquétipos) dos nossos usuários fictícios e como as notas serão distribuídas.


In [4]:
# Importação das bibliotecas necessárias

import os  # Para lidar com caminhos de arquivos do sistema
import numpy as np  # Para operações matemáticas avançadas e geração de números aleatórios (ruído)
import pandas as pd  # Para manipulação de dados em tabelas (DataFrames)

# Definimos onde os dados estão e onde o resultado será salvo.

# Como estamos no Jupyter, assumimos que tudo está na mesma pasta.

GAMES_PATH = "50_games.json"
MATRIX_PATH = "utility_matrix.csv"

SEED = 42  # Semente aleatória fixa para garantir que os resultados sejam reproduzíveis
RATING_RATE = 0.65  # Taxa de preenchimento: cada usuário avaliará cerca de 65% dos jogos (simulando que ninguém joga tudo)
N_USERS = 100  # Total de usuários que vamos simular
N_GAMES = 50  # Total de jogos na nossa base de dados
RATING_SCALE = (1, 5)  # A nota mínima é 1 e a máxima é 5

In [5]:
# PERFIS DE USUÁRIO
# Aqui criamos "personas". Cada persona tem preferências específicas.
# Se um atributo for 'None', significa que o usuário não se importa com aquele critério (aceita qualquer coisa).
PROFILES = [
    {
        "name": "Action RPG Fan",
        "genre": ["RPG", "Souls-like"],
        "perspective": ["Third-Person"],
        "category": ["Singleplayer"],
        "n": 17,  # Número de usuários que terão este exato perfil
    },
    {
        "name": "Survival Fan",
        "genre": ["Survival", "Survival Horror"],
        "perspective": None,  # Aceita 1ª pessoa, 3ª pessoa, etc.
        "category": None,  # Aceita multiplayer ou singleplayer
        "n": 16,
    },
    {
        "name": "Competitive Shooter",
        "genre": ["Shooter"],
        "perspective": ["First-Person"],
        "category": ["Multiplayer"],
        "n": 17,
    },
    {
        "name": "Multiplayer Generalist",
        "genre": None,  # Joga qualquer gênero
        "perspective": None,
        "category": [
            "Multiplayer",
            "Co-op",
        ],  # Desde que dê para jogar com outras pessoas
        "n": 15,
    },
    {
        "name": "Souls Enjoyer",
        "genre": ["Souls-like", "RPG"],
        "perspective": ["Third-Person", "2D"],
        "category": ["Singleplayer"],
        "n": 13,
    },
    {
        "name": "Simulation Fan",
        "genre": ["Simulation"],
        "perspective": None,
        "category": None,
        "n": 12,
    },
    {
        "name": "Casual Gamer",
        "genre": None,
        "perspective": None,
        "category": None,  # Sem preferência forte, joga um pouco de tudo
        "n": 10,
    },
]

In [6]:
# DISTRIBUIÇÕES DE NOTA
# Define as médias (loc) e os desvios padrões (scale) para as notas geradas.
# Usaremos uma distribuição gaussiana (curva de sino) para simular o comportamento humano real.
DISTRIBUTIONS = {
    "high": {"loc": 4.5, "scale": 0.5},  # Gosta muito: notas tendem a ficar entre 4 e 5
    "medium": {
        "loc": 3.5,
        "scale": 0.7,
    },  # Gosta um pouco: notas tendem a ficar entre 3 e 4
    "low": {
        "loc": 2.2,
        "scale": 1.0,
    },  # Não gosta muito: notas tendem a ficar entre 1 e 3
}


def _validate_config() -> None:
    """Verifica se os parâmetros definidos fazem sentido antes de rodar o código."""
    total_users = sum(p["n"] for p in PROFILES)

    # O comando 'assert' trava o código se a condição for falsa, ajudando a evitar erros invisíveis.
    assert (
        total_users == N_USERS
    ), f"Erro: A soma de usuários nos perfis é {total_users}, mas o esperado era {N_USERS}."
    assert all(
        k in DISTRIBUTIONS for k in ("high", "medium", "low")
    ), "Erro: DISTRIBUTIONS deve conter exatamente as chaves 'high', 'medium' e 'low'."


# Executa a validação logo após definir as variáveis
_validate_config()
print("Etapa 1 concluída: Configurações validadas com sucesso!")

Etapa 1 concluída: Configurações validadas com sucesso!


---

### Etapa 2: Funções Principais do Motor de Geração

Nesta etapa, construímos as engrenagens que farão o trabalho pesado. Teremos funções para:

1. **Calcular Afinidade:** Comparar as características de um jogo com o gosto de um perfil.
2. **Gerar Nota:** Transformar a afinidade em um número (1 a 5) usando probabilidade.
3. **Carregar Jogos:** Ler nosso arquivo JSON.
4. **Gerar a Matriz:** Cruzar todos os usuários com todos os jogos.
5. **Gerar Resumo:** Criar estatísticas sobre a matriz final.


In [7]:
def calculate_affinity(game: dict, profile: dict) -> str:
    """
    Calcula o nível de afinidade (high, medium, low) entre um jogo e um perfil.
    Utiliza um sistema de pontuação com pesos diferentes para cada característica.
    """
    score = 0

    # O Gênero do jogo tem o maior peso (2 pontos)
    # Se o perfil não tiver preferência de gênero (None) ou se bater, ganha os pontos.
    if profile["genre"] is None or game["genre"] in profile["genre"]:
        score += 2

    # Perspectiva (ex: Primeira pessoa) tem peso 1
    if profile["perspective"] is None or game["perspective"] in profile["perspective"]:
        score += 1

    # Categoria (ex: Multiplayer) tem peso 1
    if profile["category"] is None or game["category"] in profile["category"]:
        score += 1

    # Define a classificação final baseada na pontuação total (máximo 4)
    if score >= 3:
        return "high"  # Bateu o gênero + pelo menos 1 característica secundária
    elif score == 2:
        return "medium"  # Bateu só o gênero, ou só as duas secundárias
    else:
        return "low"  # Bateu pouca coisa ou quase nada

In [8]:
def generate_rating(affinity: str, rng: np.random.Generator) -> int:
    """
    Gera uma nota inteira (1 a 5) adicionando ruído gaussiano baseado na afinidade.
    Isso simula o fato de que um usuário pode dar nota baixa para um jogo do seu gênero
    favorito só porque não gostou da história, por exemplo.
    """
    # Pega os parâmetros matemáticos definidos na Etapa 1
    params = DISTRIBUTIONS[affinity]

    # Gera um número quebrado (float) usando a curva de sino
    rating_float = rng.normal(loc=params["loc"], scale=params["scale"])

    # Arredonda para inteiro e "corta" (clip) para garantir que não passe de 5 nem seja menor que 1
    return int(np.clip(round(rating_float), *RATING_SCALE))

In [9]:
def load_games(path: str = GAMES_PATH) -> pd.DataFrame:
    """
    Lê o arquivo de jogos e valida se ele tem as colunas corretas que o motor espera.
    """
    # Suporta tanto CSV quanto JSON, dependendo do que o usuário definir nos caminhos
    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".json"):
        df = pd.read_json(path)
    else:
        raise ValueError(f"Formato não suportado: '{path}'. Use .csv ou .json")

    # Verifica se a tabela lida contém as informações vitais para a lógica funcionar
    expected_columns = {"name", "genre", "perspective", "category"}
    missing = expected_columns - set(df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes no arquivo de jogos: {missing}")

    return df.reset_index(drop=True)

In [10]:
def generate_utility_matrix(
    games: pd.DataFrame, rating_rate: float = RATING_RATE, seed: int = SEED
) -> pd.DataFrame:
    """
    A função central: constrói a tabela cruzando Usuários (linhas) x Jogos (colunas).
    """
    # Inicializa o gerador de números aleatórios com a nossa SEED fixa
    rng = np.random.default_rng(seed)

    game_names = games["name"].tolist()
    n_games = len(game_names)
    n_users = sum(p["n"] for p in PROFILES)

    # Cria uma matriz vazia preenchida com 'NaN' (Not a Number)
    # NaN representa um jogo que o usuário ainda não jogou/avaliou
    matrix = np.full((n_users, n_games), np.nan)

    user_idx = 0  # Contador para rastrear em qual linha da matriz estamos

    # Para cada perfil (ex: Fã de RPG, Fã de Shooter)...
    for profile in PROFILES:
        # Para cada usuário dentro desse perfil...
        for _ in range(profile["n"]):

            # Calcula quantos jogos esse usuário vai avaliar (ex: 65% de 50 = ~33 jogos)
            n_to_rate = int(round(rating_rate * n_games))

            # Escolhe aleatoriamente QUAIS jogos ele vai avaliar, sem repetição (replace=False)
            rated_games = rng.choice(n_games, size=n_to_rate, replace=False)

            # Para cada jogo escolhido, calcula a nota e guarda na matriz
            for g_idx in rated_games:
                game = games.iloc[g_idx]
                affinity = calculate_affinity(game, profile)
                rating = generate_rating(affinity, rng)
                matrix[user_idx, g_idx] = rating

            # Pula para o próximo usuário (próxima linha)
            user_idx += 1

    # Cria nomes bonitinhos para as linhas (User_001, User_002, etc.)
    user_index = [f"User_{i+1:03d}" for i in range(n_users)]

    # Converte a matriz de NumPy para um DataFrame do Pandas (mais fácil de visualizar e exportar)
    return pd.DataFrame(matrix, index=user_index, columns=game_names)

In [11]:
def matrix_summary(df: pd.DataFrame) -> None:
    """Imprime estatísticas descritivas para entendermos como ficou a matriz gerada."""
    total_cells = df.size
    filled_cells = df.count().sum()  # Conta quantas células não são NaN
    sparsity = 1 - (
        filled_cells / total_cells
    )  # Esparsidade = porcentagem de células vazias

    print("=" * 45)
    print("           RESUMO DA MATRIZ DE UTILIDADE")
    print("=" * 45)
    print(f" Formato        : {df.shape[0]} usuários × {df.shape[1]} jogos")
    print(f" Notas dadas    : {filled_cells} de {total_cells} possíveis")
    print(f" Esparsidade    : {sparsity:.1%} (porcentagem de vazios)")
    print(f" Nota mínima    : {df.min().min():.0f}")
    print(f" Nota máxima    : {df.max().max():.0f}")
    print(f" Média geral    : {df.stack().mean():.2f}")
    print("=" * 45)
    print("\nTop 5 jogos mais bem avaliados (por média de nota):")
    # Calcula a média por coluna (jogo), ordena do maior pro menor e pega os 5 primeiros
    print(df.mean().sort_values(ascending=False).head(5).to_string())


def save_matrix(df: pd.DataFrame, path: str = MATRIX_PATH) -> None:
    """Salva o DataFrame resultante em um arquivo CSV no disco."""

    # index=True salva a coluna com os nomes 'User_001', etc.
    df.to_csv(path, index=True)
    print(f"\n[SUCESSO] Matriz exportada e salva em: {path}")


print("Etapa 2 concluída: Funções criadas na memória.")

Etapa 2 concluída: Funções criadas na memória.


---

### Etapa 3: Execução (Pipeline Principal)

Aqui é onde o código vai checar se o arquivo JSON existe, carregar os dados, aplicar as funções que criamos acima e exibir o resultado final na tela.


In [12]:
# 1. Verifica de forma segura se o arquivo fonte dos jogos existe no computador
if os.path.exists(GAMES_PATH):
    print("1. Arquivo de jogos encontrado. Carregando dados...")
    df_games = load_games()

    # Chama a função principal geradora passando o DataFrame de jogos
    print("2. Gerando a matriz de utilidade (Isso pode levar alguns segundos)...")
    df_utility = generate_utility_matrix(df_games)

    # Exibe o relatório de estatísticas no console
    print("\n3. Calculando métricas finais...\n")
    matrix_summary(df_utility)

    # Salva o resultado no formato CSV
    save_matrix(df_utility)
else:
    # Caso o usuário esqueça de colocar o JSON na pasta, mostramos um erro amigável
    print(f"ERRO CRÍTICO: O arquivo '{GAMES_PATH}' não foi encontrado.")
    print(
        "Por favor, certifique-se de que o arquivo JSON com os 50 jogos está salvo EXATAMENTE na mesma pasta que este notebook."
    )

1. Arquivo de jogos encontrado. Carregando dados...
2. Gerando a matriz de utilidade (Isso pode levar alguns segundos)...

3. Calculando métricas finais...

           RESUMO DA MATRIZ DE UTILIDADE
 Formato        : 100 usuários × 50 jogos
 Notas dadas    : 3200 de 5000 possíveis
 Esparsidade    : 36.0% (porcentagem de vazios)
 Nota mínima    : 1
 Nota máxima    : 5
 Média geral    : 3.51

Top 5 jogos mais bem avaliados (por média de nota):
Heritage - A Dragon's Tale    4.063492
Cyberpunk 2077                3.984848
Dead Cells                    3.968254
Swords & Bones 2              3.859375
Hollow Knight                 3.840000

[SUCESSO] Matriz exportada e salva em: utility_matrix.csv


---

# Modelo de Recomendação (KNN Content-Based)

Este módulo implementa o sistema de recomendação baseado em conteúdo, responsável por transformar os jogos em vetores numéricos, construir o perfil do usuário e retornar os itens mais similares.


---

### 1. Carregamento dos Dados

A função `load_games` lê o arquivo `50_games.json` e o converte em um `DataFrame`. Cada jogo contém:

- id
- nome
- gênero
- perspectiva
- categoria

Esse conjunto representa a base de conhecimento do sistema.


In [13]:
import os
import sys
import json
from typing import List, Dict
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def load_games(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as f:
        games = json.load(f)
    return pd.DataFrame(games)

---

### 2. Representação dos Jogos (Vetorização)

Para aplicar o modelo, é necessário converter os atributos categóricos em vetores numéricos.

#### 🔹 Construção das Features

A função `build_feature_string` concatena as características de cada jogo em uma única string: `"Shooter First-Person Multiplayer"`


In [14]:
def build_feature_string(game: pd.Series) -> str:
    genre = game["genre"].replace(" ", "_")
    perspective = game["perspective"].replace(" ", "_")
    category = game["category"].replace(" ", "_")

    return f"{genre} {perspective} {category}"

#### 🔹 Vetorização com CountVectorizer

Em seguida, é utilizado o `CountVectorizer` com parâmetro `binary=True`, o que resulta em uma representação equivalente ao **One-Hot Encoding**:

- cada termo vira uma dimensão
- presença → 1
- ausência → 0

Isso gera uma matriz de dimensão: (n_jogos x n_features) -> (50 x 12)


In [15]:
def build_tfidf_matrix(games_df: pd.DataFrame):

    games_df = games_df.copy()
    games_df["features"] = games_df.apply(build_feature_string, axis=1)

    vectorizer = CountVectorizer(binary=True, analyzer="word", token_pattern=r"[^\s]+")
    tfidf_matrix = vectorizer.fit_transform(games_df["features"])

    print(f"[TF-IDF] Vocabulário gerado: {vectorizer.get_feature_names_out()}")
    print(f"[TF-IDF] Dimensão da matriz: {tfidf_matrix.shape}  (jogos x termos únicos)")

    return tfidf_matrix, vectorizer, games_df

---

### 3. Construção do Perfil do Usuário

O perfil do usuário é construído a partir dos jogos avaliados.

#### 🔹 Estratégia

- Cada jogo avaliado é convertido para seu vetor correspondente
- Um peso é atribuído com base na nota:

| Rating | Peso |
| ------ | ---- |
| 5      | 1.0  |
| 4      | 0.75 |
| 3      | 0.5  |
| 2      | 0.25 |
| 1      | 0.0  |

- O vetor final é obtido por uma **média ponderada**

\[
p = \frac{\sum w_i \cdot x_i}{\sum w_i}
\]

- Em seguida, o vetor é normalizado (norma L2)

Isso garante que jogos melhor avaliados tenham maior influência no perfil.


In [16]:
def build_user_profile_positive_only(
    user_ratings: List[Dict],
    games_df: pd.DataFrame,
    tfidf_matrix: np.ndarray,
    min_rating: float = 3.0,
) -> np.ndarray:

    id_to_index = {str(game_id): idx for idx, game_id in enumerate(games_df["id"])}

    weighted_sum = np.zeros(tfidf_matrix.shape[1])
    sum_weights = 0.0
    games_used = []
    skipped = []
    ignored_low = []

    for entry in user_ratings:
        game_id = str(entry["id"])
        rating = entry.get("rating", min_rating)

        if game_id not in id_to_index:
            skipped.append(game_id)
            continue

        # Ex.:-> pesos: ..., 3->0.5, 4->0.8, 5->1.0
        weight = (rating - 1) / 4

        idx = id_to_index[game_id]
        game_vector = tfidf_matrix[idx].toarray().flatten()

        weighted_sum += weight * game_vector
        sum_weights += weight
        games_used.append((game_id, rating, weight))

    if skipped:
        print(f"[Perfil] Jogos não encontrados no catálogo (ignorados): {skipped}")
    if ignored_low:
        print(
            f"[Perfil] Jogos ignorados por rating baixo (<{min_rating}): {ignored_low}"
        )

    if len(games_used) == 0:
        raise ValueError(
            "Nenhum jogo com rating >= min_rating encontrado para o usuário."
        )

    if sum_weights > 0:
        user_vector = weighted_sum / sum_weights
    else:
        print("[Perfil] AVISO: soma de pesos zerada.")
        user_vector = weighted_sum

    # Normaliza o vetor resultante (norma L2 = 1)
    norm = np.linalg.norm(weighted_sum)
    if norm > 0:
        user_vector = weighted_sum / norm
    else:
        print("[Perfil] AVISO: vetor resultante zerado. Verifique os ratings/TF-IDF.")
        user_vector = weighted_sum

    print(f"[Perfil] Jogos usados (id, rating, peso_positivo):")
    for gid, r, w in games_used:
        print(f"         id={gid}  rating={r}  peso={w:.1f}")

    print("[Perfil]")
    return user_vector.reshape(1, -1)

---

### 4. Cálculo de Similaridade

A similaridade entre o usuário e os jogos é calculada utilizando **similaridade de cosseno**:

\[
\text{sim}(p, x) = \frac{p \cdot x}{\|p\| \cdot \|x\|}
\]

Essa métrica mede o ângulo entre os vetores, sendo amplamente utilizada em sistemas de recomendação baseados em conteúdo.


---

### 5. Geração das Recomendações

A função `find_knn_recommendations` realiza:

1. Cálculo da similaridade entre o vetor do usuário e todos os jogos
2. Remoção dos jogos já avaliados
3. Ordenação por similaridade decrescente
4. Seleção dos top-K itens

Esse processo equivale à aplicação do algoritmo **K-Nearest Neighbors (KNN)** no espaço vetorial de itens.


In [17]:
def find_knn_recommendations(
    user_vector: np.ndarray,
    tfidf_matrix,
    games_df: pd.DataFrame,
    user_ratings: List[Dict],
    k: int = 10,
) -> pd.DataFrame:

    # IDs já avaliados pelo usuário — serão excluidos
    rated_ids = {str(entry["id"]) for entry in user_ratings}

    # Converte a matri para array denso para o cálculo
    tfidf_dense = tfidf_matrix.toarray()

    # Calcula similaridade de cosseno entre perfil do usuário e todos os jogos
    similarities = cosine_similarity(user_vector, tfidf_dense).flatten()

    # Monta DataFrame com scores
    results = games_df.copy()
    results["similarity_score"] = similarities

    # Remove jogos já avaliados pelo usuário
    results = results[~results["id"].astype(str).isin(rated_ids)]

    # ordena por similaridade decrescente e retorna top-K
    recommendations = (
        results.sort_values("similarity_score", ascending=False)
        .head(k)
        .reset_index(drop=True)
    )

    return recommendations[
        ["id", "name", "genre", "perspective", "category", "similarity_score"]
    ]

---

### 6. Classe Principal do Sistema

A classe `KNNRecommender` encapsula todo o pipeline:

- `fit()`:
  - carrega os dados
  - constrói a matriz de features

- `recommend()`:
  - recebe avaliações do usuário
  - constrói o perfil
  - retorna recomendações

No código de produção ela funciona como um Singleton. Isso permite reutilização simples do modelo no backend da aplicação


In [18]:
BASE_DIR = Path().resolve()
GAMES_FILE_PATH = BASE_DIR / "50_games.json"


class KNNRecommender:
    """
    Singleton
    classe principal do sistema de recomendação KNN content-based.
    """

    def __init__(self, k: int = 15):
        self.games_filepath = GAMES_FILE_PATH
        self.games_df = None
        self.tfidf_matrix = None
        self.vectorizer = None
        self._fitted = False
        self.k = k

    def fit(self):
        """Carrega os dados e treina o TF-IDF (base de conhecimento)"""
        self.games_df = load_games(self.games_filepath)
        self.tfidf_matrix, self.vectorizer, self.games_df = build_tfidf_matrix(
            self.games_df
        )
        self._fitted = True
        print(f"[KNNRecommender] Pronto. {len(self.games_df)} jogos carregados.")
        return self

    def recommend(self, user_ratings: List[Dict]):
        if not self._fitted:
            raise RuntimeError("Chame fit() antes de recommend().")

        user_vector = build_user_profile_positive_only(
            user_ratings, self.games_df, self.tfidf_matrix
        )

        recs_df = find_knn_recommendations(
            user_vector, self.tfidf_matrix, self.games_df, user_ratings, k=self.k
        )

        return recs_df.to_dict(orient="records")

In [19]:
recommender = KNNRecommender()
recommender.fit()

# simulçao: dados mock
user_ratings = [
    {"id": "227300", "name": "Euro Truck Simulator 2", "rating": 5},
    {"id": "292030", "name": "The Witcher 3: Wild Hunt", "rating": 4},
    {
        "id": "814380",
        "name": "Sekiro™: Shadows Die Twice - GOTY Edition",
        "rating": 1,
    },
    {"id": "730", "name": "Counter-Strike 2", "rating": 4},
    {"id": "365670", "name": "Blender", "rating": 1},
]

recommendations = recommender.recommend(user_ratings)

print(f"\n=== TOP 15 JOGOS RECOMENDADOS ===")
print(pd.DataFrame(recommendations).to_string(index=False))

[TF-IDF] Vocabulário gerado: ['2d' 'co-op' 'first-person' 'multiplayer' 'rpg' 'shooter' 'simulation'
 'singleplayer' 'souls-like' 'survival' 'survival_horror' 'third_person']
[TF-IDF] Dimensão da matriz: (50, 12)  (jogos x termos únicos)
[KNNRecommender] Pronto. 50 jogos carregados.
[Perfil] Jogos usados (id, rating, peso_positivo):
         id=227300  rating=5  peso=1.0
         id=292030  rating=4  peso=0.8
         id=814380  rating=1  peso=0.0
         id=730  rating=4  peso=0.8
         id=365670  rating=1  peso=0.0
[Perfil]

=== TOP 15 JOGOS RECOMENDADOS ===
     id                            name           genre  perspective     category  similarity_score
 236390                     War Thunder      Simulation Third Person  Multiplayer          0.848528
 553850                   HELLDIVERS™ 2         Shooter Third Person  Multiplayer          0.801388
2246340            Monster Hunter Wilds             RPG Third Person  Multiplayer          0.801388
1063730             New World